# IT2011 - Artificial Intelligence and Machine Learning
## Progress Review I: Data Preprocessing & Exploratory Data Analysis (EDA)
### Group ID: `2026-Y2-S1-MET-23`
### Member 6: Silva A.M.K.N. (IT25103132)
### Assigned Technique: Feature Extraction (TF-IDF) & Dimensionality Reduction (TruncatedSVD / LSA)

---
### 1. Technique Overview & Academic Justification
Raw text cannot be fed directly into mathematical models; it must be converted into numerical vector spaces:
1. **TF-IDF Vectorization (Term Frequency - Inverse Document Frequency):**
   $$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \log\left(\frac{1 + |D|}{1 + |\{d \in D : t \in d\}|}\right) + 1$$
   Penalizes words that occur across all reviews while boosting distinctive discriminative tokens.
2. **The Curse of Dimensionality:** An n-gram vocabulary easily generates $>50,000$ sparse features, causing severe computational bottlenecks and overfitting.
3. **TruncatedSVD (Latent Semantic Analysis - LSA):** Standard PCA fails on large sparse matrices because centering dense zeros consumes gigabytes of RAM. TruncatedSVD operates directly on sparse CSR matrices, decomposing the high-dimensional term-document matrix into low-dimensional orthogonal semantic concepts.

**Viva Objective:** Explain TF-IDF mathematical formulation, demonstrate TruncatedSVD matrix reduction, and present cumulative explained variance and 2D semantic projection.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

os.makedirs('../results/eda_visualizations', exist_ok=True)
sns.set_theme(style="whitegrid", palette="muted")


### 2. Loading Dataset & Preparing Text Corpus

In [ ]:
DATA_PATH = '../data/raw/Movies_Reviews_modified_version1.csv'
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} reviews.")

# Clean simple string fill
df['Reviews_clean'] = df['Reviews'].fillna('').astype(str)


### 3. Implementing TF-IDF Vectorization with N-Grams

In [ ]:
# Sample a representative slice of 10,000 reviews for fast SVD visualization
sample_df = df.sample(n=10000, random_state=42).copy()

tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    stop_words='english',
    min_df=3,
    max_df=0.85
)

tfidf_matrix = tfidf_vectorizer.fit_transform(sample_df['Reviews_clean'])
print(f"TF-IDF Sparse Matrix Shape: {tfidf_matrix.shape[0]} reviews x {tfidf_matrix.shape[1]} vocabulary features")
print(f"Sparsity: {(1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])) * 100:.2f}% zero entries")


### 4. Dimensionality Reduction via TruncatedSVD (Latent Semantic Analysis)

In [ ]:
# Apply TruncatedSVD with 50 components
N_COMPONENTS = 50
svd = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)
svd_matrix = svd.fit_transform(tfidf_matrix)

explained_variance = svd.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

print(f"Variance explained by top 2 components: {cumulative_variance[1]*100:.2f}%")
print(f"Variance explained by top {N_COMPONENTS} components: {cumulative_variance[-1]*100:.2f}%")

sample_df['LSA_1'] = svd_matrix[:, 0]
sample_df['LSA_2'] = svd_matrix[:, 1]


### 5. Individual EDA Visualizations (Viva Presentation Requirement)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left Plot: Cumulative Explained Variance Ratio
axes[0].plot(range(1, N_COMPONENTS + 1), cumulative_variance, marker='o', color='#8e44ad', linewidth=2)
axes[0].set_title('TruncatedSVD Cumulative Explained Variance Ratio', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Latent Semantic Components', fontsize=11)
axes[0].set_ylabel('Cumulative Explained Variance', fontsize=11)
axes[0].grid(True, linestyle='--', alpha=0.7)

# Right Plot: 2D LSA Projection by Emotion Class
# Filter to 4 primary contrasting emotions for visual clarity
focus_emotions = ['joy', 'sadness', 'anger', 'fear']
plot_subset = sample_df[sample_df['emotion'].isin(focus_emotions)]

sns.scatterplot(
    data=plot_subset, x='LSA_1', y='LSA_2', hue='emotion',
    palette={'joy': '#2ecc71', 'sadness': '#3498db', 'anger': '#e74c3c', 'fear': '#f39c12'},
    alpha=0.6, s=25, ax=axes[1]
)
axes[1].set_title('2D Latent Semantic Projection (LSA 1 vs LSA 2)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Latent Concept 1 (General Movie Vocabulary)', fontsize=11)
axes[1].set_ylabel('Latent Concept 2 (Sentiment & Emotion Polarity)', fontsize=11)
axes[1].legend(title='Emotion', loc='upper right')

plt.tight_layout()
output_plot_path = '../results/eda_visualizations/member6_tfidf_svd_variance.png'
plt.savefig(output_plot_path, dpi=300, bbox_inches='tight')
print(f"EDA plot saved successfully to: {output_plot_path}")
plt.show()


### 6. Key Findings & Viva Talking Points (For Silva A.M.K.N.)

> **Viva Preparation Notes:**
> 1. **Why use TruncatedSVD instead of standard PCA on text?**
>    PCA requires mean-centering of columns ($X - \mu$). Centering turns a 99.8% sparse matrix into a 100% dense matrix of floating-point numbers, instantly exceeding RAM memory limits. TruncatedSVD computes the singular value decomposition directly on sparse CSR matrices without densification.
> 2. **What does Sublinear TF do?**
>    Replacing raw term frequency $TF$ with $1 + \log(TF)$ prevents a review that repeats a word 20 times from exerting 20x more weight than a review mentioning it once.
> 3. **What does the 2D projection reveal?**
>    Latent Component 1 captures dominant movie review syntax, while Component 2 distinctly separates positive emotions (`joy`) from negative emotions (`sadness`, `anger`, `fear`).
